# Specification Mining

## What This Is
Specification mining learns formal properties from execution traces, enabling verification of systems where a human-written specification is unavailable.

**Temporal logic (LTL)**: Linear Temporal Logic expresses properties over time:
- G(p): p is true *globally* (always)
- F(p): p is true *eventually* (at some point)
- G(p → F(q)): globally, whenever p holds, q eventually follows
- G(p → X(q)): globally, whenever p holds, q holds in the *next* step

**Mining from traces**: Given a set of execution traces, learn LTL formulas that:
1. Are *consistent* with all traces (no trace violates the formula)
2. Are *informative* (not trivially true)
3. Are *concise* (short formulas preferred)

In [ ]:
import numpy as np
from typing import List, Dict, Tuple

np.random.seed(42)

# LTL property mining from execution traces
# Traces are sequences of state observations

# Generate traces from a simple system
def generate_safe_trace(length: int) -> List[Dict]:
    """Safe system: sensor reading stays in [0, 1], then actuator resets."""
    trace = []
    state = 0.3
    for t in range(length):
        noise = np.random.normal(0, 0.05)
        state = np.clip(state + 0.1 + noise, 0, 1)
        actuator = 1 if state > 0.8 else 0
        if actuator == 1:
            state = np.random.uniform(0.1, 0.3)  # reset
        trace.append({'sensor': round(state, 3), 'actuator': actuator,
                       'alarm': int(state > 0.9)})
    return trace

traces = [generate_safe_trace(20) for _ in range(50)]

# Mine properties: check candidate LTL templates
def check_G_implies_eventually(traces, cond_fn, effect_fn, max_steps=5):
    """Check: G(cond -> F[0..max_steps](effect))"""
    violations = 0
    total = 0
    for trace in traces:
        for t in range(len(trace)):
            if cond_fn(trace[t]):
                total += 1
                found = any(effect_fn(trace[t2]) 
                            for t2 in range(t, min(t+max_steps, len(trace))))
                if not found:
                    violations += 1
    return violations, total

# Candidate properties to mine
candidates = [
    ('G(sensor>0.8 -> F(actuator=1))',
     lambda s: s['sensor'] > 0.8,
     lambda s: s['actuator'] == 1),
    ('G(alarm=1 -> F(sensor<0.5))',
     lambda s: s['alarm'] == 1,
     lambda s: s['sensor'] < 0.5),
    ('G(actuator=1 -> F(sensor<0.4))',
     lambda s: s['actuator'] == 1,
     lambda s: s['sensor'] < 0.4),
    ('G(sensor<0.2 -> F(sensor>0.5))',
     lambda s: s['sensor'] < 0.2,
     lambda s: s['sensor'] > 0.5),
]

print('Specification Mining Results:')
print('=' * 60)
for name, cond, effect in candidates:
    viols, total = check_G_implies_eventually(traces, cond, effect)
    hold_rate = (total - viols) / total if total > 0 else 0
    status = 'MINED' if hold_rate > 0.95 else 'REJECTED'
    print(f'  [{status}] {name}')
    print(f'           hold_rate={hold_rate:.3f}, total_triggers={total}')
